In [1]:
import pandas as pd
import numpy as np
import duckdb

import plotly.express as px

In [4]:
df_base = pd.read_json("./data/dataset.json")
df_base

,tournament_id,match_id,stage_name,match_date,team,confederation,rank,host,matches_played,lst_match_days_ago,score,win,draw,opp_team,opp_confederation,opp_rank,opp_host,opp_lst_match_days_ago
0,WC-1998,M-1998-02,group stage,897436800,Morocco,CAF,13,0,1,0,2,0,1,Norway,UEFA,7,0,0
1,WC-1998,M-1998-02,group stage,897436800,Norway,UEFA,7,0,1,0,2,0,1,Morocco,CAF,13,0,0
2,WC-1998,M-1998-04,group stage,897523200,Austria,UEFA,31,0,1,0,1,0,1,Cameroon,CAF,49,0,0
3,WC-1998,M-1998-04,group stage,897523200,Cameroon,CAF,49,0,1,0,1,0,1,Austria,UEFA,31,0,0
4,WC-1998,M-1998-05,group stage,897609600,Paraguay,CONMEBOL,29,0,1,0,0,0,1,Bulgaria,UEFA,35,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
843,WC-2022,M-2022-62,semi-finals,1670976000,France,UEFA,4,0,6,4,2,1,0,Morocco,CAF,22,0,4
844,WC-2022,M-2022-63,third-place match,1671235200,Morocco,CAF,22,0,7,3,1,0,0,Croatia,UEFA,12,0,4
845,WC-2022,M-2022-63,third-place match,1671235200,Croatia,UEFA,12,0,7,4,2,1,0,Morocco,CAF,22,0,3
846,WC-2022,M-2022-64,final,1671321600,Argentina,CONMEBOL,3,0,7,5,3,1,0,France,UEFA,4,0,4


In [5]:
dfs = duckdb.query("""
SELECT * FROM df_base WHERE tournament_id = 'WC-2022' AND team = 'Argentina'
""").df()
dfs

,tournament_id,match_id,stage_name,match_date,team,confederation,rank,host,matches_played,lst_match_days_ago,score,win,draw,opp_team,opp_confederation,opp_rank,opp_host,opp_lst_match_days_ago
0,WC-2022,M-2022-05,group stage,1669075200,Argentina,CONMEBOL,3,0,1,0,1,0,0,Saudi Arabia,AFC,51,0,0
1,WC-2022,M-2022-24,group stage,1669420800,Argentina,CONMEBOL,3,0,2,4,2,1,0,Mexico,CONCACAF,13,0,4
2,WC-2022,M-2022-39,group stage,1669766400,Argentina,CONMEBOL,3,0,3,4,2,1,0,Poland,UEFA,26,0,4
3,WC-2022,M-2022-50,round of 16,1670025600,Argentina,CONMEBOL,3,0,4,3,2,1,0,Australia,AFC,38,0,3
4,WC-2022,M-2022-58,quarter-finals,1670544000,Argentina,CONMEBOL,3,0,5,6,2,1,0,Netherlands,UEFA,8,0,6
5,WC-2022,M-2022-61,semi-finals,1670889600,Argentina,CONMEBOL,3,0,6,4,3,1,0,Croatia,UEFA,12,0,4
6,WC-2022,M-2022-64,final,1671321600,Argentina,CONMEBOL,3,0,7,5,3,1,0,France,UEFA,4,0,4


 * Rank / Current Rank / Ongoing Rank (Rank Adv, Match Q, Form Delta)
 * Offensive Strenght
 * Defensive Strenght
 * Confederation matchups
 * Rest

### Rank

In [6]:
df_base[['rank', 'opp_rank', 'score', 'win']]

,rank,opp_rank,score,win
0,13,7,2,0
1,7,13,2,0
2,31,49,1,0
3,49,31,1,0
4,29,35,0,0
...,...,...,...,...
843,4,22,2,1
844,22,12,1,0
845,12,22,2,1
846,3,4,3,1


In [7]:
# Bucketize the ranks
df = df_base.copy()
l = 100
b = 8
lbls = [f'R:{i}-{i+(b-1)}' for i in range(1, l-b, b)]
df['team_rank_bucket'] = pd.cut(df['rank'], bins=np.arange(1, l, b), right=False, labels=lbls)
df['opp_rank_bucket'] = pd.cut(df['opp_rank'], bins=np.arange(1, l, b), right=False, labels=lbls)
df['win_draw'] = df.win + 0.5 * df.draw

# Reworked to be a heatmap showing home win probability
fig = px.density_heatmap(
    df,
    x='team_rank_bucket',
    y='opp_rank_bucket',
    z='win_draw',
    text_auto=".0%", # Format text in cells as percentages
    histfunc='avg',
    title='Win Probability: Team Rank vs. Opp Team Rank (Bucketed)',
    labels={'color': 'Win Probability'},
    color_continuous_scale='RdYlGn', # Red-Yellow-Green scale
    template='plotly_dark',
    category_orders={'team_rank_bucket': lbls, 'opp_rank_bucket': lbls}
)

fig.update_layout(
    xaxis_title="Team Rank (Bucketed)",
    yaxis_title="Opp Team Rank (Bucketed)",
    coloraxis_showscale=False,
    showlegend=False,
    width=800,
    height=800,
    font=dict(
        family="Courier New, monospace",
        size=12,
        color="white"
    )
)

fig.show()

In [8]:
# Bucketize the ranks
df = df_base.copy()
l = 100
b = 8
lbls = [f'R:{i}-{i+(b-1)}' for i in range(1, l-b, b)]
df['team_rank_bucket'] = pd.cut(df['rank'], bins=np.arange(1, l, b), right=False, labels=lbls)
df['opp_rank_bucket'] = pd.cut(df['opp_rank'], bins=np.arange(1, l, b), right=False, labels=lbls)
df['win_draw'] = df.win + 0.5 * df.draw

# Reworked to be a heatmap showing home win probability
fig = px.density_heatmap(
    df,
    x='team_rank_bucket',
    y='opp_rank_bucket',
    z='score',
    text_auto='.2f',
    histfunc='avg',
    title='Avg Goals by Team: Team Rank vs. Opp Team Rank (Bucketed)',
    labels={'color': 'Win Probability'},
    color_continuous_scale='RdYlGn', # Red-Yellow-Green scale
    template='plotly_dark',
    category_orders={'team_rank_bucket': lbls, 'opp_rank_bucket': lbls}
)

fig.update_layout(
    xaxis_title="Team Rank (Bucketed)",
    yaxis_title="Opp Team Rank (Bucketed)",
    coloraxis_showscale=False,
    showlegend=False,
    width=800,
    height=800,
    font=dict(
        family="Courier New, monospace",
        size=12,
        color="white"
    )
)

fig.show()

In [9]:
# 1. Calculate the rank difference.
df = df_base.copy()
df['rank_diff'] = df['opp_rank'] - df['rank']

# 2. Bucketize the rank difference into intervals of size 4.
# We define the edges of the bins from -32 to +32 with a step of 4.
b_size = 8
max_abs_diff = 50 # df['rank_diff'].abs().max()
# Ensure bins cover the full range of data, stepping by 4
bin_edge = int(np.ceil(max_abs_diff / b_size) * b_size)
bins = np.arange(-bin_edge, bin_edge + 1, b_size)

# Create meaningful labels for each bucket.
labels = [f'{i} to {i + b_size - 1}' for i in bins[:-1]]

df['rank_diff_bucket'] = pd.cut(
    df['rank_diff'],
    bins=bins,
    labels=labels,
    right=False, # Intervals are [left, right)
    include_lowest=True
)

# 3. Group by the rank difference bucket and calculate home win probability.
# We use observed=False to include all possible buckets in the output, even if empty.
win_prob_by_bucket = (
    df.groupby('rank_diff_bucket', observed=False)['score']
    .mean()
    .reset_index(name='mean_score')
)

# 4. Create the bar plot.
fig = px.bar(
    win_prob_by_bucket,
    x='rank_diff_bucket',
    y='mean_score',
    title='Avg. Goals Scored vs. Bucketed Rank Difference',
    labels={
        'Rank_Diff_Bucket': 'Rank Difference Bucket (Opp Team Rank - Team Rank)',
        'mean_score': 'Avg. Goals Scored'
    },
    template='plotly_dark'
)

# 5. Update the layout for better readability and aesthetics.
fig.update_layout(
    xaxis_title="Rank Difference Bucket (Opp Team Rank - Team Rank)",
    yaxis_title="Goals Scored",
    yaxis=dict(
        #tickformat=".0%",  # Format y-axis labels as percentages
        #range=[0, 1]       # Set y-axis range from 0 to 1
    ),
    font=dict(
        family="Courier New, monospace",
        size=12,
        color="white"
    )
)

fig.show()

In [10]:
# 1. Calculate the rank difference.
df = df_base.copy()
df['rank_diff'] = np.log(df['opp_rank']) - np.log(df['rank'])

# 2. Bucketize the rank difference into intervals of size 4.
# We define the edges of the bins from -32 to +32 with a step of 4.
b_size = 0.5
max_abs_diff = 3 # df['rank_diff'].abs().max()
# Ensure bins cover the full range of data, stepping by 4
bin_edge = int(np.ceil(max_abs_diff / b_size) * b_size)
bins = np.arange(-bin_edge, bin_edge + 1, b_size)

# Create meaningful labels for each bucket.
labels = [f'{i + b_size - 1} to {i} ' for i in bins[:-1]]

df['rank_diff_bucket'] = pd.cut(
    df['rank_diff'],
    bins=bins,
    labels=labels,
    right=False, # Intervals are [left, right)
    include_lowest=True
)

# 3. Group by the rank difference bucket and calculate home win probability.
# We use observed=False to include all possible buckets in the output, even if empty.
win_prob_by_bucket = (
    df.groupby('rank_diff_bucket', observed=False)['score']
    .mean()
    .reset_index(name='mean_score')
)

# 4. Create the bar plot.
fig = px.bar(
    win_prob_by_bucket,
    x='rank_diff_bucket',
    y='mean_score',
    title='Avg. Goals Scored vs. Bucketed Rank Difference',
    labels={
        'Rank_Diff_Bucket': 'Rank Difference Bucket (Opp Team Rank - Opp Team Rank)',
        'mean_score': 'Avg. Goals Scored'
    },
    template='plotly_dark'
)

# 5. Update the layout for better readability and aesthetics.
fig.update_layout(
    xaxis_title="Rank Difference Bucket (Home Team - Away Team)",
    yaxis_title="Goals Scored",
    yaxis=dict(
        #tickformat=".0%",  # Format y-axis labels as percentages
        #range=[0, 1]       # Set y-axis range from 0 to 1
    ),
    font=dict(
        family="Courier New, monospace",
        size=12,
        color="white"
    )
)

fig.show()

In [11]:
import seaborn as sns
import statsmodels.api as sm
import matplotlib.pyplot as plt
import numpy as np

In [12]:
# Calculate rank differences
df['rank_diff'] = df['opp_rank'] - df['rank']
df['log_rank_diff'] = np.log(df['opp_rank']) - np.log(df['rank'])
df['win_exp'] = 1 / (1 + (df['rank'] / df['opp_rank'])**0.625 )

In [14]:
# Filter out rows with missing score or rank info if any
model_df = df.dropna(subset=['score', 'rank_diff', 'log_rank_diff'])

# Poisson Regression - Linear Rank Diff
X1 = sm.add_constant(model_df['rank_diff'])
y = model_df['score']
model1 = sm.GLM(y, X1, family=sm.families.Poisson()).fit()

# Poisson Regression - Log Rank Diff
X2 = sm.add_constant(model_df['log_rank_diff'])
model2 = sm.GLM(y, X2, family=sm.families.Poisson()).fit()

# Poisson Regression - Win Exp
X3 = sm.add_constant(model_df['win_exp'])
model3 = sm.GLM(y, X3, family=sm.families.Poisson()).fit()

print("Model 1 (Linear Rank Diff) AIC:", model1.aic)
print("Model 2 (Log Rank Diff) AIC:", model2.aic)
print("Model 3 (Win Exp) AIC:", model3.aic)

print("\n--- Model 1 Summary ---\n")
print(model1.summary())
print("\n--- Model 2 Summary ---\n")
print(model2.summary())
print("\n--- Model 3 Summary ---\n")
print(model3.summary())

Model 1 (Linear Rank Diff) AIC: 2414.4666079844064
Model 2 (Log Rank Diff) AIC: 2423.7648468887564
Model 3 (Win Exp) AIC: 2419.2393375526635

--- Model 1 Summary ---

                 Generalized Linear Model Regression Results                  
Dep. Variable:                  score   No. Observations:                  848
Model:                            GLM   Df Residuals:                      846
Model Family:                 Poisson   Df Model:                            1
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1205.2
Date:                Sat, 14 Feb 2026   Deviance:                       987.99
Time:                        15:28:44   Pearson chi2:                     890.
No. Iterations:                     5   Pseudo R-squ. (CS):             0.1071
Covariance Type:            nonrobust                                         
                 coef    std err          z

In [15]:
import plotly.graph_objects as go

# Calculate predictions for all models
model_df['pred_m1'] = model1.predict(X1)
model_df['pred_m2'] = model2.predict(X2)
model_df['pred_m3'] = model3.predict(X3)

# Group by rank_diff to get average actual score and average predicted scores
# restricting the range to avoid noisy outliers at extremes
# filter out extreme values for cleaner plot if needed, but plotting all for now
plot_df = model_df.groupby('rank_diff')[['score', 'pred_m1', 'pred_m2', 'pred_m3']].mean()

# Create figure
fig = go.Figure()

# Add traces
fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['score'], name='True Value', mode='markers', marker=dict(color='white', size=4, opacity=0.6)))
fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['pred_m1'], name='Model 1 (Linear Rank Diff)', mode='lines', line=dict(color='cyan')))
fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['pred_m2'], name='Model 2 (Log Rank Diff)', mode='lines', line=dict(color='magenta')))
fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['pred_m3'], name='Model 3 (Win Exp)', mode='lines', line=dict(color='yellow')))

# Update layout with dark theme
fig.update_layout(
    title='Model Comparison: Average Predicted vs Actual Score by Rank Difference',
    xaxis_title='Rank Difference (Opponent Rank - Team Rank)',
    yaxis_title='Average Score',
    template='plotly_dark',
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
    font=dict(
        family="Courier New, monospace",
        size=12,
        color="white"
    )
)

fig.show()

In [16]:
def calculate_rank_shift(rank_a, rank_b, result, shape=1.5, k_mul=5):
    # Ensure ranks are never <= 0 to prevent complex numbers in the power function
    r_a = max(1.0, rank_a)
    r_b = max(1.0, rank_b)
    
    # 1. Expected Result
    expected_a = 1 / (1 + (r_a / r_b)**shape)
    
    # 2. Calculate the 'Shift'
    k_factor = np.log1p(abs(r_a - r_b)) * k_mul
    shift = k_factor * (expected_a - result)
    
    return shift

def calculate_current_rank(rank, t_id, m_date, team, df_base, shape=1.1, k_mul=5):
    past_games = duckdb.execute(f"""
        SELECT g.*
        FROM df_base as g
        WHERE g.tournament_id = $1 AND g.match_date < $2 AND g.team = $3
        ORDER BY g.match_date
        """, [t_id, int(m_date), team]
    ).df()

    if len(past_games) == 0:
        return rank
    
    team_rank = rank
    for i, g in past_games.iterrows():
        opp_team = g['opp_team']
        opp_base_rank = g['opp_rank']
        p_m_game = g['match_date']
        opp_rank = calculate_current_rank(opp_base_rank, t_id, p_m_game, opp_team, df_base, shape, k_mul)
        # print (f"Team: {team} [R:{team_rank}] / Opp Team: {opp_team} [R:{opp_rank}]")

        shifts = calculate_rank_shift(team_rank, opp_rank, float(g['win'] + 0.5 * g['draw']), shape, k_mul)
        team_rank = max(1.,round(team_rank + shifts, 1))
        # print (f"   Team -> Old Rank: {round(team_rank - shifts, 1)} -> New Rank: {team_rank}")
    
    return team_rank


In [18]:
def process_tournament_history(df_base, shape=1.5, k_mul=5):
    # Sort matches chronologically to ensure ranks evolve correctly
    df_sorted = df_base.sort_values(['tournament_id', 'match_date', 'team', 'opp_team']).copy()
    
    # Storage for the 'Live Ranks' and the resulting columns
    # Key format: (tournament_id, team_name)
    current_ranks = {}
    rank_at_kickoff = []
    opp_rank_at_kickoff = []
    rank_after_match = []

    for i, row in df_sorted.iterrows():
        t_id = row['tournament_id']
        t_name = row['team']
        o_name = row['opp_team']
        
        # 1. Get current rank for Team A
        # If not in dict, use the 'rank' column from the row
        r_a = current_ranks.get((t_id, t_name), row['rank'])
        
        # 2. Get current rank for Opponent
        # If not in dict, use the 'opp_rank' column from the row
        r_b = current_ranks.get((t_id, o_name), row['opp_rank'])
        
        # Save kickoff ranks for features
        rank_at_kickoff.append(r_a)
        opp_rank_at_kickoff.append(r_b)
        
        # 3. Calculate Shift
        result = float(row['win'] + 0.5 * row['draw'])
        shift = calculate_rank_shift(r_a, r_b, result, shape, k_mul)
        
        # 4. Update the Rank (Moving toward 1 for a win, away for a loss)
        new_rank = max(1.0, round(r_a + shift, 1))
        
        # Update state for this team
        current_ranks[(t_id, t_name)] = new_rank

    # Add the results back to the dataframe
    df_sorted['cur_rank'] = rank_at_kickoff
    df_sorted['opp_cur_rank'] = opp_rank_at_kickoff
    
    return df_sorted

In [19]:
df_rv = process_tournament_history(df_base, shape=1.5, k_mul=5)

In [21]:
duckdb.query("""
SELECT * FROM df_rv WHERE tournament_id = 'WC-2022' AND team = 'France'
""").df()

,tournament_id,match_id,stage_name,match_date,team,confederation,rank,host,matches_played,lst_match_days_ago,score,win,draw,opp_team,opp_confederation,opp_rank,opp_host,opp_lst_match_days_ago,cur_rank,opp_cur_rank
0,WC-2022,M-2022-08,group stage,1669075200,France,UEFA,4,0,1,0,4,1,0,Australia,AFC,38,0,0,4.0,38.6
1,WC-2022,M-2022-23,group stage,1669420800,France,UEFA,4,0,2,4,2,1,0,Denmark,UEFA,10,0,4,3.4,16.4
2,WC-2022,M-2022-38,group stage,1669766400,France,UEFA,4,0,3,4,0,0,0,Tunisia,CAF,30,0,4,2.3,31.0
3,WC-2022,M-2022-51,round of 16,1670112000,France,UEFA,4,0,4,4,3,1,0,Poland,UEFA,26,0,4,18.9,23.1
4,WC-2022,M-2022-60,quarter-finals,1670630400,France,UEFA,4,0,5,6,2,1,0,England,UEFA,5,0,6,15.4,15.1
5,WC-2022,M-2022-62,semi-finals,1670976000,France,UEFA,4,0,6,4,2,1,0,Morocco,CAF,22,0,4,14.7,8.8
6,WC-2022,M-2022-64,final,1671321600,France,UEFA,4,0,7,4,3,0,0,Argentina,CONMEBOL,3,0,5,8.1,5.3


In [34]:
# Comprehensive comparison of all methodologies
import plotly.graph_objects as go

# Calculate all features
df_comparison = df_rv.copy()

# 1. Linear rank differences (already have these)
df_comparison['rank_diff'] = df_comparison['opp_rank'] - df_comparison['rank']
df_comparison['cur_rank_diff'] = df_comparison['opp_cur_rank'] - df_comparison['cur_rank']

# 2. Log rank differences
df_comparison['log_rank_diff'] = np.log(df_comparison['opp_rank']) - np.log(df_comparison['rank'])
df_comparison['log_cur_rank_diff'] = np.log(df_comparison['opp_cur_rank']) - np.log(df_comparison['cur_rank'])

# 3. Win expectation formulas
df_comparison['win_exp_base'] = 1 / (1 + (df_comparison['rank'] / df_comparison['opp_rank'])**0.625)
df_comparison['win_exp_cur'] = 1 / (1 + (df_comparison['cur_rank'] / df_comparison['opp_cur_rank'])**0.625)

# Fit Poisson regression models for all 6 approaches
y = df_comparison['score']

# 1. Linear rank differences
X1_base = sm.add_constant(df_comparison[['rank_diff']])
X1_cur = sm.add_constant(df_comparison[['cur_rank_diff']])
model1_base = sm.GLM(y, X1_base, family=sm.families.Poisson()).fit()
model1_cur = sm.GLM(y, X1_cur, family=sm.families.Poisson()).fit()

# 2. Log rank differences
X2_base = sm.add_constant(df_comparison[['log_rank_diff']])
X2_cur = sm.add_constant(df_comparison[['log_cur_rank_diff']])
model2_base = sm.GLM(y, X2_base, family=sm.families.Poisson()).fit()
model2_cur = sm.GLM(y, X2_cur, family=sm.families.Poisson()).fit()

# 3. Win expectation
X3_base = sm.add_constant(df_comparison[['win_exp_base']])
X3_cur = sm.add_constant(df_comparison[['win_exp_cur']])
model3_base = sm.GLM(y, X3_base, family=sm.families.Poisson()).fit()
model3_cur = sm.GLM(y, X3_cur, family=sm.families.Poisson()).fit()

# Bucketize for cleaner visualization
bucket_size = 5

# For rank_diff approaches, bucket by rank difference
max_rank_diff = max(abs(df_comparison['rank_diff'].min()), abs(df_comparison['rank_diff'].max()),
                     abs(df_comparison['cur_rank_diff'].min()), abs(df_comparison['cur_rank_diff'].max()))
bin_edge = int(np.ceil(max_rank_diff / bucket_size) * bucket_size)
bins_rank = np.arange(-bin_edge, bin_edge + bucket_size, bucket_size)

df_comparison['rank_diff_bucket'] = pd.cut(df_comparison['rank_diff'], bins=bins_rank, right=False)
df_comparison['cur_rank_diff_bucket'] = pd.cut(df_comparison['cur_rank_diff'], bins=bins_rank, right=False)
df_comparison['rank_diff_mid'] = df_comparison['rank_diff_bucket'].apply(lambda x: x.mid if pd.notna(x) else np.nan)
df_comparison['cur_rank_diff_mid'] = df_comparison['cur_rank_diff_bucket'].apply(lambda x: x.mid if pd.notna(x) else np.nan)

# For log rank diff
max_log_diff = max(abs(df_comparison['log_rank_diff'].min()), abs(df_comparison['log_rank_diff'].max()),
                    abs(df_comparison['log_cur_rank_diff'].min()), abs(df_comparison['log_cur_rank_diff'].max()))
bucket_size_log = 0.5
bin_edge_log = int(np.ceil(max_log_diff / bucket_size_log) * bucket_size_log)
bins_log = np.arange(-bin_edge_log, bin_edge_log + bucket_size_log, bucket_size_log)

df_comparison['log_rank_diff_bucket'] = pd.cut(df_comparison['log_rank_diff'], bins=bins_log, right=False)
df_comparison['log_cur_rank_diff_bucket'] = pd.cut(df_comparison['log_cur_rank_diff'], bins=bins_log, right=False)
df_comparison['log_rank_diff_mid'] = df_comparison['log_rank_diff_bucket'].apply(lambda x: x.mid if pd.notna(x) else np.nan)
df_comparison['log_cur_rank_diff_mid'] = df_comparison['log_cur_rank_diff_bucket'].apply(lambda x: x.mid if pd.notna(x) else np.nan)

# For win expectation (0 to 1 range)
bucket_size_win = 0.1
bins_win = np.arange(0, 1.1, bucket_size_win)

df_comparison['win_exp_base_bucket'] = pd.cut(df_comparison['win_exp_base'], bins=bins_win, right=False)
df_comparison['win_exp_cur_bucket'] = pd.cut(df_comparison['win_exp_cur'], bins=bins_win, right=False)
df_comparison['win_exp_base_mid'] = df_comparison['win_exp_base_bucket'].apply(lambda x: x.mid if pd.notna(x) else np.nan)
df_comparison['win_exp_cur_mid'] = df_comparison['win_exp_cur_bucket'].apply(lambda x: x.mid if pd.notna(x) else np.nan)

# Aggregate bucketed data
rank_diff_bucketed = df_comparison.groupby('rank_diff_mid', observed=False)['score'].mean().reset_index().dropna()
cur_rank_diff_bucketed = df_comparison.groupby('cur_rank_diff_mid', observed=False)['score'].mean().reset_index().dropna()
log_rank_diff_bucketed = df_comparison.groupby('log_rank_diff_mid', observed=False)['score'].mean().reset_index().dropna()
log_cur_rank_diff_bucketed = df_comparison.groupby('log_cur_rank_diff_mid', observed=False)['score'].mean().reset_index().dropna()
win_exp_base_bucketed = df_comparison.groupby('win_exp_base_mid', observed=False)['score'].mean().reset_index().dropna()
win_exp_cur_bucketed = df_comparison.groupby('win_exp_cur_mid', observed=False)['score'].mean().reset_index().dropna()

# Generate prediction curves
# For rank diff (map to common x-axis)
x_rank_range = np.linspace(-bin_edge, bin_edge, 100)
pred1_base = model1_base.predict(sm.add_constant(pd.DataFrame({'rank_diff': x_rank_range})))
pred1_cur = model1_cur.predict(sm.add_constant(pd.DataFrame({'cur_rank_diff': x_rank_range})))

# For log rank diff  
x_log_range = np.linspace(-bin_edge_log, bin_edge_log, 100)
pred2_base = model2_base.predict(sm.add_constant(pd.DataFrame({'log_rank_diff': x_log_range})))
pred2_cur = model2_cur.predict(sm.add_constant(pd.DataFrame({'log_cur_rank_diff': x_log_range})))

# For win expectation
x_win_range = np.linspace(0, 1, 100)
pred3_base = model3_base.predict(sm.add_constant(pd.DataFrame({'win_exp_base': x_win_range})))
pred3_cur = model3_cur.predict(sm.add_constant(pd.DataFrame({'win_exp_cur': x_win_range})))

# Create comprehensive comparison figure with 3 subplots
from plotly.subplots import make_subplots

fig_comp = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        f'Linear Rank Difference<br><sub>Base AIC: {model1_base.aic:.1f} | Current AIC: {model1_cur.aic:.1f}</sub>',
        f'Log Rank Difference<br><sub>Base AIC: {model2_base.aic:.1f} | Current AIC: {model2_cur.aic:.1f}</sub>', 
        f'Win Expectation<br><sub>Base AIC: {model3_base.aic:.1f} | Current AIC: {model3_cur.aic:.1f}</sub>'
    ),
    horizontal_spacing=0.08
)

# Subplot 1: Linear Rank Difference
fig_comp.add_trace(go.Scatter(x=rank_diff_bucketed['rank_diff_mid'], y=rank_diff_bucketed['score'],
                               mode='markers', name='Actual (Base Rank)',
                               marker=dict(color='cyan', size=8, opacity=0.6, symbol='circle'),
                               legendgroup='base', showlegend=True), row=1, col=1)
fig_comp.add_trace(go.Scatter(x=x_rank_range, y=pred1_base, mode='lines',
                               name='Predicted (Base Rank)',
                               line=dict(color='cyan', width=2.5), legendgroup='base', showlegend=True), row=1, col=1)

fig_comp.add_trace(go.Scatter(x=cur_rank_diff_bucketed['cur_rank_diff_mid'], y=cur_rank_diff_bucketed['score'],
                               mode='markers', name='Actual (Current Rank)',
                               marker=dict(color='magenta', size=8, opacity=0.6, symbol='square'),
                               legendgroup='cur', showlegend=True), row=1, col=1)
fig_comp.add_trace(go.Scatter(x=x_rank_range, y=pred1_cur, mode='lines',
                               name='Predicted (Current Rank)',
                               line=dict(color='magenta', width=2.5, dash='dash'), legendgroup='cur', showlegend=True), row=1, col=1)

# Subplot 2: Log Rank Difference
fig_comp.add_trace(go.Scatter(x=log_rank_diff_bucketed['log_rank_diff_mid'], y=log_rank_diff_bucketed['score'],
                               mode='markers', name='Actual (Base Rank)',
                               marker=dict(color='cyan', size=8, opacity=0.6, symbol='circle'),
                               legendgroup='base', showlegend=False), row=1, col=2)
fig_comp.add_trace(go.Scatter(x=x_log_range, y=pred2_base, mode='lines',
                               name='Predicted (Base Rank)',
                               line=dict(color='cyan', width=2.5), legendgroup='base', showlegend=False), row=1, col=2)

fig_comp.add_trace(go.Scatter(x=log_cur_rank_diff_bucketed['log_cur_rank_diff_mid'], y=log_cur_rank_diff_bucketed['score'],
                               mode='markers', name='Actual (Current Rank)',
                               marker=dict(color='magenta', size=8, opacity=0.6, symbol='square'),
                               legendgroup='cur', showlegend=False), row=1, col=2)
fig_comp.add_trace(go.Scatter(x=x_log_range, y=pred2_cur, mode='lines',
                               name='Predicted (Current Rank)',
                               line=dict(color='magenta', width=2.5, dash='dash'), legendgroup='cur', showlegend=False), row=1, col=2)

# Subplot 3: Win Expectation
fig_comp.add_trace(go.Scatter(x=win_exp_base_bucketed['win_exp_base_mid'], y=win_exp_base_bucketed['score'],
                               mode='markers', name='Actual (Base Rank)',
                               marker=dict(color='cyan', size=8, opacity=0.6, symbol='circle'),
                               legendgroup='base', showlegend=False), row=1, col=3)
fig_comp.add_trace(go.Scatter(x=x_win_range, y=pred3_base, mode='lines',
                               name='Predicted (Base Rank)',
                               line=dict(color='cyan', width=2.5), legendgroup='base', showlegend=False), row=1, col=3)

fig_comp.add_trace(go.Scatter(x=win_exp_cur_bucketed['win_exp_cur_mid'], y=win_exp_cur_bucketed['score'],
                               mode='markers', name='Actual (Current Rank)',
                               marker=dict(color='magenta', size=8, opacity=0.6, symbol='square'),
                               legendgroup='cur', showlegend=False), row=1, col=3)
fig_comp.add_trace(go.Scatter(x=x_win_range, y=pred3_cur, mode='lines',
                               name='Predicted (Current Rank)',
                               line=dict(color='magenta', width=2.5, dash='dash'), legendgroup='cur', showlegend=False), row=1, col=3)

# Update axes
fig_comp.update_xaxes(title_text="Rank Diff (Opp - Team)", row=1, col=1)
fig_comp.update_xaxes(title_text="Log Rank Diff", row=1, col=2)
fig_comp.update_xaxes(title_text="Win Expectation", row=1, col=3)
fig_comp.update_yaxes(title_text="Avg Goals", row=1, col=1)

# Update layout
fig_comp.update_layout(
    title_text='Comprehensive Methodology Comparison: Predicting Team Goals',
    template='plotly_dark',
    height=600,
    width=1600,
    font=dict(family="Courier New, monospace", size=11, color="white"),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.25,
        xanchor="center",
        x=0.5,
        font=dict(size=10)
    ),
    margin=dict(b=100),
    hovermode='x unified'
)

# Print summary statistics
print("="*80)
print("MODEL COMPARISON SUMMARY")
print("="*80)
print("\n1. LINEAR RANK DIFFERENCE")
print(f"   Base Rank:    AIC = {model1_base.aic:.2f}")
print(f"   Current Rank: AIC = {model1_cur.aic:.2f} (Δ = {model1_base.aic - model1_cur.aic:+.2f})")
print("\n2. LOG RANK DIFFERENCE")
print(f"   Base Rank:    AIC = {model2_base.aic:.2f}")
print(f"   Current Rank: AIC = {model2_cur.aic:.2f} (Δ = {model2_base.aic - model2_cur.aic:+.2f})")
print("\n3. WIN EXPECTATION")
print(f"   Base Rank:    AIC = {model3_base.aic:.2f}")
print(f"   Current Rank: AIC = {model3_cur.aic:.2f} (Δ = {model3_base.aic - model3_cur.aic:+.2f})")
print("\n" + "="*80)
print(f"BEST MODEL: ", end="")
all_aics = {
    'Linear Base': model1_base.aic, 'Linear Current': model1_cur.aic,
    'Log Base': model2_base.aic, 'Log Current': model2_cur.aic,
    'WinExp Base': model3_base.aic, 'WinExp Current': model3_cur.aic
}
best_model = min(all_aics, key=all_aics.get)
print(f"{best_model} (AIC = {all_aics[best_model]:.2f})")
print("="*80)

fig_comp.show()

MODEL COMPARISON SUMMARY

1. LINEAR RANK DIFFERENCE
   Base Rank:    AIC = 2414.47
   Current Rank: AIC = 2393.61 (Δ = +20.86)

2. LOG RANK DIFFERENCE
   Base Rank:    AIC = 2423.76
   Current Rank: AIC = 2397.94 (Δ = +25.82)

3. WIN EXPECTATION
   Base Rank:    AIC = 2419.24
   Current Rank: AIC = 2391.64 (Δ = +27.59)

BEST MODEL: WinExp Current (AIC = 2391.64)
